In [35]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.anthropic import AnthropicChatCompletionClient
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

In [55]:
load_dotenv(override=True)

True

In [56]:
@dataclass
class Message:
    content: str

In [57]:
class SimpleAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("Simple")
    @message_handler
    async def on_my_message(self, message: Message, ctx: MessageContext) -> Message:
        return Message(content=f"This is {self.id.type}-{self.id.key}. You said '{message.content}' and I disagree")

In [58]:
runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())

AgentType(type='simple_agent')

In [59]:
runtime.start()

In [60]:
agent_id = AgentId("simple_agent", "default")
response = await runtime.send_message(Message("Well, hi there!"), agent_id)
print(">>>", response.content)

>>> This is simple_agent-default. You said 'Well, hi there!' and I disagree


In [61]:
await runtime.stop()
await runtime.close()

In [62]:
class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        openaimodel_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent("LLMAgent", model_client=openaimodel_client)

    @message_handler
    async def handle_my_message(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received message: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type} responded: {reply}")
        return Message(content=reply)

In [63]:
runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())
await MyLLMAgent.register(runtime, "LLMAgent", lambda: MyLLMAgent())

AgentType(type='LLMAgent')

In [64]:
runtime.start()
response = await runtime.send_message(Message("Well, hi there!"), AgentId("LLMAgent", "default"))
print(">>>", response.content)
response = await runtime.send_message(Message(response.content), AgentId("simple_agent", "default"))
print(">>>", response.content)

LLMAgent received message: Well, hi there!
LLMAgent responded: Hello! How can I assist you today?
>>> Hello! How can I assist you today?
>>> This is simple_agent-default. You said 'Hello! How can I assist you today?' and I disagree


In [65]:
await runtime.stop()
await runtime.close()

In [66]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        openaimodel_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=openaimodel_client)

    @message_handler
    async def handle_my_message(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        anthropicmodel_client = AnthropicChatCompletionClient(model="claude-sonnet-4-5", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=anthropicmodel_client)

    @message_handler
    async def handle_my_message(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [67]:
JUDGE = "You are judging a game of rock, paper scissors, lizard, spock. The players have made their choices:\n"

class RPSLSAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        judgemodel_client = OpenAIChatCompletionClient(model="gpt-4", temperature=1.0)
        self.delegate = AssistantAgent(name, model_client=judgemodel_client)

    @message_handler
    async def handle_my_message(self, message: Message, ctx: MessageContext) -> Message:
        instruction = "You are playing rock, paper, scissors, lizard, spock. Respond only with one word, one of the following: rock, paper, scissors, lizard or spock."
        message = Message(content=instruction)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)
        response2 = await self.send_message(message, inner_2)
        result = f"Player 1: {response1.content}\n Player 2: {response2.content}"
        judgement = f"{JUDGE}: {result}Who wins?"
        message = TextMessage(content=judgement, source="user")
        response = await self.delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)

In [68]:
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RPSLSAgent.register(runtime, "rock_paper_scissors_lizard_spock", lambda: RPSLSAgent("rock_paper_scissors_lizard_spock"))
runtime.start()

In [70]:
agent_id = AgentId("rock_paper_scissors_lizard_spock", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(">>>", response.content)

>>> Player 1: spock
 Player 2: paper

TERMINATEPlayer 2 wins. According to the rules of the game, Paper disproves Spock. Therefore, the Paper (Player 2) defeats Spock (Player 1). 

TERMINATE
